# Module 4: Đối ngẫu trong tối ưu hóa — code

Notebook đi kèm site *Tối ưu hóa phi tuyến & lồi*. Chạy tuần tự từng ô (Colab có sẵn numpy, scipy, sympy, cvxpy). Tự giải tay trước rồi đối chiếu kết quả.

## LP đối ngẫu: giải cả hai bài và so giá trị (dạng A, B)

Bài 4.1: max 3x1+2x2. Gốc và đối ngẫu phải cho cùng giá trị 9; y chính là giá bóng.

In [1]:
import numpy as np
from scipy.optimize import linprog
A = np.array([[1, 1], [2, 1.]]); b = np.array([4, 5.]); c = np.array([3, 2.])
p = linprog(-c, A_ub=A, b_ub=b, bounds=[(0, None)] * 2, method="highs")
d = linprog(b, A_ub=-A.T, b_ub=-c, bounds=[(0, None)] * 2, method="highs")
print("x* =", p.x, " p* =", -p.fun)
print("y* =", d.x, " d* =", d.fun)
print("giá bóng (HiGHS):", -p.ineqlin.marginals)

x* = [1. 3.]  p* = 9.0
y* = [1. 1.]  d* = 9.0
giá bóng (HiGHS): [1. 1.]


## Giá bóng và độ nhạy: đổi vế phải b2 (dạng A)

Đổi nguyên liệu 5 → 6: lợi nhuận tăng đúng y2 = 1 trong khoảng [4, 8].

In [2]:
import numpy as np
from scipy.optimize import linprog
A = np.array([[1, 1], [2, 1.]]); c = np.array([3, 2.])
for t in (3, 4, 5, 6, 8, 9):
    r = linprog(-c, A_ub=A, b_ub=[4, t], bounds=[(0, None)] * 2, method="highs")
    print("b2 =", t, " p* =", round(-r.fun, 4), " x* =", r.x.round(3))

b2 = 3  p* = 6.0  x* = [0. 3.]
b2 = 4  p* = 8.0  x* = [0. 4.]
b2 = 5  p* = 9.0  x* = [1. 3.]
b2 = 6  p* = 10.0  x* = [2. 2.]
b2 = 8  p* = 12.0  x* = [ 4. -0.]
b2 = 9  p* = 12.0  x* = [4. 0.]


## Hàm đối ngẫu Boyd 5.1 và độ nhạy (dạng D)

g(lambda) = 1 + 8 lambda − 9 lambda²/(1+lambda): cực đại tại 2 với giá trị 5.

In [3]:
import numpy as np
lam = np.linspace(0, 10, 100001); g = 1 + 8*lam - 9*lam**2/(1 + lam)
i = g.argmax(); print("lambda* =", lam[i], " d* =", round(g[i], 6))
pu = lambda u: (3 - np.sqrt(1 + u))**2 + 1
h = 1e-6; print("dp*/du(0) =", round((pu(h) - pu(-h))/(2*h), 5), "(= -lambda*)")

lambda* = 2.0  d* = 5.0
dp*/du(0) = -2.0 (= -lambda*)


## Bài code · Lập đối ngẫu LP theo bảng quy tắc

Đầu vào: $c,A,b$, dấu hàng (&gt;=, =, &lt;=) và dấu cột (&gt;=0, free, &lt;=0). Đầu ra: giá trị gốc, giá trị đối ngẫu và trạng thái.

In [4]:
import numpy as np
from scipy.optimize import linprog
def primal(c, A, b, rows, cols):
    A = np.array(A, float); Aub, bub, Aeq, beq = [], [], [], []
    for i, s in enumerate(rows):
        if s == ">=": Aub.append(-A[i]); bub.append(-b[i])
        elif s == "<=": Aub.append(A[i]); bub.append(b[i])
        else: Aeq.append(A[i]); beq.append(b[i])
    bd = [(0, None) if s == ">=0" else ((None, 0) if s == "<=0" else (None, None)) for s in cols]
    return linprog(c, A_ub=Aub or None, b_ub=bub or None, A_eq=Aeq or None, b_eq=beq or None, bounds=bd, method="highs")
def dual(c, A, b, rows, cols):
    A = np.array(A, float); Aub, bub, Aeq, beq = [], [], [], []
    for j, s in enumerate(cols):
        if s == ">=0": Aub.append(A[:, j]); bub.append(c[j])
        elif s == "<=0": Aub.append(-A[:, j]); bub.append(-c[j])
        else: Aeq.append(A[:, j]); beq.append(c[j])
    bd = [(0, None) if s == ">=" else ((None, 0) if s == "<=" else (None, None)) for s in rows]
    return linprog(-np.array(b, float), A_ub=Aub or None, b_ub=bub or None, A_eq=Aeq or None, b_eq=beq or None, bounds=bd, method="highs")
c, A, b = [1, -3, 4, 0], [[2, 3, -3, 1], [1, 1, 1, 0], [3, 5, -5, -1]], [12, 4, 6]
p = primal(c, A, b, ["="] * 3, [">=0"] * 4); d = dual(c, A, b, ["="] * 3, [">=0"] * 4)
print("gốc", p.fun, " đối ngẫu", -d.fun)

gốc -5.875  đối ngẫu -5.875


## Bài code · Kiểm điều kiện bù

Đầu vào: $x,y$ và dữ liệu LP chuẩn tắc. Đầu ra: hai nhóm tích bù và kết luận tối ưu.

In [5]:
import numpy as np
def comp(c, A, b, x, y, tol=1e-9):
    A = np.array(A, float); x = np.array(x, float); y = np.array(y, float)
    r = y * (A @ x - b); s = x * (np.array(c) - A.T @ y)
    feas = (A @ x >= np.array(b) - tol).all() and (x >= -tol).all() and (A.T @ y <= np.array(c) + tol).all() and (y >= -tol).all()
    return r.round(9), s.round(9), feas and abs(r).max() < tol and abs(s).max() < tol
print(comp([3, 2], [[1, 1], [2, 1]], [4, 5], [1, 3], [1, 1]))

(array([0., 0.]), array([0., 0.]), np.True_)
